# CV5570 · Raj · CUDA COLMAP + PatchMatch MVS baseline (site data)

The classical arm of the comparison. Replaces the CPU stereo workaround with COLMAP's real
PatchMatch MVS, on exactly the same images and camera poses the GS runs use.

**Setup:** Accelerator **GPU T4 ×2** · Internet **On** · Add Input → `cv5570-site1` →
**Save Version → Save & Run All**. Runtime ~20 min (conda route) or ~60 min (source build).

**When done:** Output tab → download `deliverables_patchmatch.zip` → attach in the Claude chat with
"patchmatch results".

In [ ]:
# ---- 1. GPU check -------------------------------------------------------------
import subprocess, sys, os, glob, json, time, shutil
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU. Settings (right panel) -> Accelerator -> GPU T4 x2"
N_GPUS = torch.cuda.device_count()
print(f"{N_GPUS} GPU(s) visible")

In [ ]:
# ---- 2. locate the team dataset (cv5570-site1) -----------------------------------
import glob, os, shutil, subprocess
hits = glob.glob("/kaggle/input/**/colmap_workspace/sparse/cameras.bin", recursive=True)
assert hits, ("Dataset not attached. Right panel -> Add Input -> search 'cv5570-site1' -> Add. "
              "Then re-run this cell.")
SRC = os.path.dirname(os.path.dirname(hits[0]))           # .../colmap_workspace
DATA = "/kaggle/working/data"
if not os.path.exists(DATA):
    shutil.copytree(SRC, DATA)                           # writable copy
n_img = sum(len(f) for _, _, f in os.walk(f"{DATA}/images"))
print("dataset:", SRC, "->", DATA, "|", n_img, "images")
print(sorted(os.listdir(DATA)))

In [ ]:
# ---- install COLMAP with CUDA (the apt build has NO CUDA - that is what blocked us in Sept)
import subprocess, os, shutil
def sh(cmd, check=True, tail=1500):
    print("$", cmd[:200]); r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode and check:
        print(r.stdout[-tail:], r.stderr[-tail:]); raise RuntimeError("command failed")
    return r
def has_cuda(binary):
    r = subprocess.run(f"{binary} -h", shell=True, capture_output=True, text=True)
    return "with CUDA" in (r.stdout + r.stderr)

COLMAP = None
# Option A - conda-forge CUDA build via micromamba (~5 min)
try:
    sh("cd /kaggle/working && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba")
    sh("/kaggle/working/bin/micromamba create -y -q -p /kaggle/working/mm -c conda-forge 'colmap=*=*cuda*'")
    if has_cuda("/kaggle/working/mm/bin/colmap"):
        COLMAP = "/kaggle/working/mm/bin/colmap"
except Exception as e:
    print("Option A failed:", e)

# Option B - build COLMAP 3.9.1 from source with CUDA (~40 min)
if COLMAP is None:
    print("Falling back to source build ...")
    cc = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True,
                        capture_output=True, text=True).stdout.split()[0].replace(".", "")
    sh("apt-get update -qq && apt-get install -y -qq git cmake ninja-build build-essential "
       "libboost-program-options-dev libboost-filesystem-dev libboost-graph-dev libboost-system-dev "
       "libeigen3-dev libflann-dev libfreeimage-dev libmetis-dev libgoogle-glog-dev libgtest-dev "
       "libgmock-dev libsqlite3-dev libglew-dev qtbase5-dev libqt5opengl5-dev libcgal-dev libceres-dev")
    if not os.path.exists("/kaggle/working/colmap_src"):
        sh("git clone -q --depth 1 --branch 3.9.1 https://github.com/colmap/colmap.git /kaggle/working/colmap_src")
    sh(f"cd /kaggle/working/colmap_src && mkdir -p build && cd build && cmake .. -GNinja "
       f"-DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES={cc} -DGUI_ENABLED=OFF > cmake.log 2>&1 "
       f"&& ninja -j4 > ninja.log 2>&1", tail=4000)
    COLMAP = sh("find /kaggle/working/colmap_src/build -name colmap -type f -perm -u+x | head -1").stdout.strip()

assert COLMAP and has_cuda(COLMAP), "No CUDA COLMAP - paste the output above to Claude"
os.environ["QT_QPA_PLATFORM"] = "offscreen"
print("CUDA COLMAP ready:", COLMAP)
print(subprocess.run(f"{COLMAP} -h", shell=True, capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
# ---- 4. PatchMatch MVS + fusion on the site data -----------------------------------
import time, os, json
WS = "/kaggle/working/data"            # the shared dataset IS a COLMAP dense workspace
OUT = "/kaggle/working/deliverables"; os.makedirs(OUT, exist_ok=True)
t0 = time.time()
sh(f"{COLMAP} patch_match_stereo --workspace_path {WS} --workspace_format COLMAP "
   f"--PatchMatchStereo.geom_consistency true --PatchMatchStereo.max_image_size 1024 "
   f"> {OUT}/patchmatch.log 2>&1", tail=4000)
t_pm = time.time() - t0
sh(f"{COLMAP} stereo_fusion --workspace_path {WS} --workspace_format COLMAP "
   f"--input_type geometric --output_path {WS}/fused.ply > {OUT}/fusion.log 2>&1", tail=4000)
t_all = time.time() - t0
print(f"PatchMatch {t_pm/60:.1f} min, total {t_all/60:.1f} min")
json.dump({"patchmatch_min": t_pm/60, "total_min": t_all/60}, open(f"{OUT}/timing_patchmatch.json", "w"))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plyfile'])

In [ ]:
# ---- 5. slim export (binary PLY xyz+rgb, same format as the GS exports) ------------
import numpy as np, os, json, shutil
WS = "/kaggle/working/data"; OUT = "/kaggle/working/deliverables"
from plyfile import PlyData
v = PlyData.read(f"{WS}/fused.ply")["vertex"]
xyz = np.stack([v["x"], v["y"], v["z"]], 1).astype(np.float32)
rgb = np.stack([v["red"], v["green"], v["blue"]], 1).astype(np.uint8)
n_total = len(xyz); print(f"fused: {n_total:,} points")
MAX = 1_500_000
if n_total > MAX:
    idx = np.random.default_rng(0).choice(n_total, MAX, replace=False); xyz, rgb = xyz[idx], rgb[idx]
dt = np.dtype([("x","<f4"),("y","<f4"),("z","<f4"),("red","u1"),("green","u1"),("blue","u1")])
a = np.empty(len(xyz), dt); a["x"], a["y"], a["z"] = xyz.T; a["red"], a["green"], a["blue"] = rgb.T
with open(f"{OUT}/site1_patchmatch_fused.ply", "wb") as f:
    f.write((f"ply\nformat binary_little_endian 1.0\nelement vertex {len(a)}\nproperty float x\nproperty float y\n"
             "property float z\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n").encode())
    f.write(a.tobytes())
json.dump({"fused_points_total": int(n_total)}, open(f"{OUT}/stats_patchmatch.json", "w"))
shutil.make_archive("/kaggle/working/deliverables_patchmatch", "zip", OUT)
for fn in sorted(os.listdir(OUT)):
    print(f"{fn:40s} {os.path.getsize(f'{OUT}/{fn}')/1e6:7.1f} MB")
print("ZIP -> /kaggle/working/deliverables_patchmatch.zip")